# 🟦 Pattern 5 — Memory / Stateful Agents

> **One-line definition:** the agent keeps state **across** runs — it has identity over time.

⚠️ **Important framing:** memory is usually an **augmentation**, not a standalone architecture.
A ReAct agent + memory = a *memory-enabled ReAct agent*. Memory is a **capability you bolt on**
to Patterns 1–4 and 6–7.

---

## 1. The three layers of memory (the mental model)

```
┌──────────────────────────────────────────────────────┐
│ 1. WITHIN-RUN            state["messages"]           │
│    lives: one invoke()                               │
│    "what did the tool just return?"                  │
├──────────────────────────────────────────────────────┤
│ 2. SHORT-TERM (thread)   CHECKPOINTER                │
│    lives: one conversation (thread_id)               │
│    "what did the user say 3 turns ago?"              │
├──────────────────────────────────────────────────────┤
│ 3. LONG-TERM (cross-thread)  STORE                   │
│    lives: forever, scoped by user_id                 │
│    "this user is vegetarian and lives in Cologne"    │
└──────────────────────────────────────────────────────┘
```

---

## 2. Checkpointer vs Store — the table to memorise

| | **Checkpointer** | **Store** |
|---|---|---|
| Scope | one `thread_id` | across all threads |
| Stores | the **whole graph state** | app-defined **JSON documents** |
| Keyed by | `thread_id` | namespace tuple, e.g. `("memories", user_id)` |
| Written by | ⚙️ **LangGraph, automatically** | 👤 **you, explicitly** in a node |
| Read by | ⚙️ LangGraph, automatically | 👤 you, explicitly (`store.search`) |
| Analogy | session / conversation | user profile / CRM |
| Dev impl | `InMemorySaver()` | `InMemoryStore()` |
| Prod impl | `PostgresSaver` | `PostgresStore` |
| One-liner | turns a graph into a **conversation** | turns a conversation into a **relationship** |

👉 **You almost always need both.** Passing only one is the most common architecture mistake.

---

## 3. Key Properties (pointwise)

| Property | Memory agent |
|---|---|
| Planning | 🟡 inherited from base pattern |
| Loop | 🟡 inherited |
| Memory | ✅ **the defining feature** |
| Identity over time | ✅ yes |
| Extra infra | ✅ needs a DB in production |
| Main risk | 🔴 context window growth + stale facts |

---

## 4. The confusing parts (resolved 👇)

### Q1: "I added a checkpointer but the agent still forgets. Why?"

Almost always **one of these three**:

1. ❌ You forgot `config={"configurable": {"thread_id": "..."}}` — without it you get an error
   or a fresh thread every call.
2. ❌ You used a **new** `thread_id` each turn — each one is a separate conversation.
3. ❌ Your state key has **no reducer**, so each node overwrites instead of appending.
   Use `Annotated[list, add_messages]`.

---

### Q2: "Why do I pass the full message list AND have a checkpointer? Isn't that double?"

No — they do different jobs:

- **State** (`messages`) is the *working set* for the current run.
- **Checkpointer** *persists* that state between runs so the next `invoke()` starts with it
  already loaded.

You pass only the **new** message. LangGraph loads the old ones from the checkpoint and
`add_messages` appends yours to them.

```python
# Turn 2 — you only send the NEW message:
graph.invoke({"messages": [HumanMessage("What's my name?")]}, config)
# LangGraph internally: [old msgs from checkpoint] + [your new msg]
```

---

### Q3: "How does `store` get into my node? I never passed it."

**Dependency injection.** If your node function declares a parameter named `store`,
LangGraph injects the store you passed to `compile(store=...)`:

```python
def node(state, config, *, store: BaseStore):   # <- name must be `store`
```

Same for `config`. This is why the parameter names are not arbitrary.

---

### Q4: "Namespace vs key — what's the difference?"

Think **directory** vs **filename**:

```python
store.put(("memories", "user-42"), "diet", {"value": "vegetarian"})
#          └────── namespace ────┘   └key┘   └────── value ──────┘
```

- **Namespace** = a tuple, scopes/isolates. `("memories", user_id)` keeps users apart.
- **Key** = the document id within that namespace.

**Key-choice rule:**
| Data type | Key strategy | Why |
|---|---|---|
| Attributes that **change** (diet, city, plan) | **deterministic** (`"diet"`) | new value overwrites old |
| Records that **accumulate** (past issues, notes) | **UUID** | each is a distinct record |

---

### Q5: "My context window explodes after 50 turns. Now what?"

Inevitable. Three strategies, use them **before** you need them:

| Strategy | How | Trade-off |
|---|---|---|
| **Trim** | `trim_messages(max_tokens=...)` | cheap, but hard-loses old info |
| **Summarise** | a node that condenses old turns into one message | keeps gist, costs an LLM call |
| **Extract to Store** | pull facts out, drop the raw turns | best long-term, most work |

---

## 5. Graph we will build

```
   START
     │
     ▼
 ┌───────────────┐
 │ load_memories │  store.search(("memories", user_id))
 └───────────────┘
     │
     ▼
 ┌───────────────┐
 │     agent     │  LLM sees: memories + thread history + new msg
 └───────────────┘
     │
     ▼
 ┌───────────────┐
 │ save_memories │  store.put(...)  <- explicit write
 └───────────────┘
     │
     ▼
    END

 (checkpointer works INVISIBLY underneath the whole thing)
```


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [1]:
# --- Standard setup used by every notebook in this series ---
import os, getpass

def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready")

LLM ready


---

# 🟦 PART A — Short-term memory (Checkpointer)

## 6. The bare minimum

Three things. That's it.


In [2]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


def chat_node(state: MessagesState) -> dict:
    """A plain chat node. Notice: NOTHING here is memory-aware.
    The checkpointer does its work outside the node."""
    return {"messages": [llm.invoke(state["messages"])]}


b = StateGraph(MessagesState)
b.add_node("chat", chat_node)
b.add_edge(START, "chat")
b.add_edge("chat", END)

# ⭐ (1) create a checkpointer
checkpointer = InMemorySaver()

# ⭐ (2) pass it to compile()
chat_graph = b.compile(checkpointer=checkpointer)

# ⭐ (3) ALWAYS pass a thread_id — this is the conversation's routing key
config = {"configurable": {"thread_id": "conv-1"}}

# Turn 1
r1 = chat_graph.invoke({"messages": [HumanMessage("Hi, I'm Ankit and I live in Cologne.")]}, config)
print("A:", r1["messages"][-1].content)

# Turn 2 — we send ONLY the new message; history is restored from the checkpoint
r2 = chat_graph.invoke({"messages": [HumanMessage("Where do I live?")]}, config)
print("A:", r2["messages"][-1].content)

print(f"\nMessages now in state: {len(r2['messages'])}  <- history accumulated")

A: Hallo Ankit! Nice to meet you. Cologne is a beautiful city in Germany, known for its rich history, cultural landmarks, and of course, the famous Cologne Cathedral. What do you like to do in your free time, and what's your favorite thing about living in Cologne?
A: You live in Cologne, Germany.

Messages now in state: 4  <- history accumulated


### Threads are isolated — proof


In [3]:
other = {"configurable": {"thread_id": "conv-2"}}   # a DIFFERENT thread
r = chat_graph.invoke({"messages": [HumanMessage("Where do I live?")]}, other)
print("Thread conv-2:", r["messages"][-1].content[:160])
print("\n^ It has no idea. Different thread_id == different conversation.")

Thread conv-2: I don't have that information. I'm a large language model, I don't have the ability to know your personal details or location. I can only respond based on the t

^ It has no idea. Different thread_id == different conversation.


---

## 7. Inspecting & time-travelling

The checkpointer saves a snapshot **after every node**. That gives you debugging superpowers.


In [5]:
# Current snapshot
snap = chat_graph.get_state(config)
print("Messages in thread:", len(snap.values["messages"]))
print("Next node to run   :", snap.next, "(empty = finished)")

# Full history, newest first
history = list(chat_graph.get_state_history(config))
print(f"\nCheckpoints saved: {len(history)}")
for h in history[:4]:
    print(f"  {h.config['configurable']['checkpoint_id'][:8]}... "
          f"| msgs={len(h.values.get('messages', []))} | next={h.next}")

Messages in thread: 4
Next node to run   : () (empty = finished)

Checkpoints saved: 6
  1f181367... | msgs=4 | next=()
  1f181367... | msgs=3 | next=('chat',)
  1f181367... | msgs=2 | next=('__start__',)
  1f181367... | msgs=2 | next=()


### ⏪ Time travel — resume from an old checkpoint

Pass a `checkpoint_id` and the graph forks reality from that point.


In [6]:
if len(history) > 2:
    old = history[2]                       # some earlier state
    fork_cfg = old.config                  # contains the checkpoint_id
    out = chat_graph.invoke({"messages": [HumanMessage("What's my name?")]}, fork_cfg)
    print("Forked branch says:", out["messages"][-1].content[:160])

Forked branch says: Your name is Ankit.


---

## 8. Production checkpointers

| Backend | Import | Use for |
|---|---|---|
| `InMemorySaver` | `langgraph.checkpoint.memory` | notebooks, tests |
| `SqliteSaver` | `langgraph.checkpoint.sqlite` | single-process prototypes |
| `PostgresSaver` | `langgraph.checkpoint.postgres` | ✅ production |
| `AsyncPostgresSaver` | same | ✅ async production |

```python
# Prototype (survives restarts, single process only — file locking!)
from langgraph.checkpoint.sqlite import SqliteSaver
with SqliteSaver.from_conn_string("./memory.db") as cp:
    graph = builder.compile(checkpointer=cp)

# Production
from langgraph.checkpoint.postgres import PostgresSaver
with PostgresSaver.from_conn_string("postgresql://...") as cp:
    cp.setup()                 # ⚠️ run ONCE to create tables
    graph = builder.compile(checkpointer=cp)
```

⚠️ **`thread_id` must be < 255 chars** — Postgres stores it in a bounded column. Use a UUID.


---

# 🟩 PART B — Long-term memory (Store)

Now for the part **you** must write explicitly.


In [ ]:
import uuid
from typing import List
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field


class Memories(BaseModel):
    """Facts worth persisting about a user."""
    facts: List[str] = Field(
        description="Durable facts about the user (preferences, location, constraints). "
                    "Empty list if the message contains nothing worth remembering."
    )

extractor = llm.with_structured_output(Memories)

### Node 1 — Load memories

**⭐ The magic:** declaring a parameter literally named `store` makes LangGraph inject it.
Same for `config`. The names are not arbitrary.


In [ ]:
def load_memories(state: MessagesState, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Read this user's long-term facts and inject them as a SystemMessage."""
    user_id = config["configurable"].get("user_id", "anonymous")
    namespace = ("memories", user_id)      # namespace = directory, scoped per user

    # .search() returns Items. With a real vector store, `query=` does semantic search.
    items = store.search(namespace, query=state["messages"][-1].content, limit=10)

    if not items:
        return {}                          # nothing to inject; return an EMPTY update

    facts = "\n".join(f"- {i.value['text']}" for i in items)
    print(f"  📖 loaded {len(items)} memories for {user_id}")

    # Prepend context. add_messages appends, so this lands before the user's new turn
    # only if we're careful — for simplicity we just append; the LLM still reads it.
    return {"messages": [SystemMessage(content=f"What you know about this user:\n{facts}")]}

### Node 2 — The agent (unchanged from Pattern 2!)


In [ ]:
def memory_agent(state: MessagesState) -> dict:
    """Ordinary agent. It just happens to see memories in its message list."""
    return {"messages": [llm.invoke(state["messages"])]}

### Node 3 — Save memories

**Deterministic key vs UUID** — the decision that bites people later:
- `store.put(ns, "diet", {...})` → overwrites. Good for *changing attributes*.
- `store.put(ns, str(uuid4()), {...})` → accumulates. Good for *event records*.


In [ ]:
def save_memories(state: MessagesState, config: RunnableConfig, *, store: BaseStore) -> dict:
    """Extract durable facts from the recent turns and persist them."""
    user_id = config["configurable"].get("user_id", "anonymous")
    namespace = ("memories", user_id)

    convo = "\n".join(f"{m.type}: {m.content}" for m in state["messages"][-4:])
    result = extractor.invoke(
        f"Extract durable facts about the USER from this exchange. "
        f"Ignore small talk and anything transient.\n\n{convo}"
    )

    for fact in result.facts:
        # UUID key -> facts accumulate rather than overwrite each other
        store.put(namespace, str(uuid.uuid4()), {"text": fact})
        print(f"  💾 saved: {fact}")

    return {}     # writes went to the STORE, not to state -> no state update needed

### Wire it up — **both** checkpointer and store


In [ ]:
mb = StateGraph(MessagesState)
mb.add_node("load", load_memories)
mb.add_node("agent", memory_agent)
mb.add_node("save", save_memories)

mb.add_edge(START, "load")
mb.add_edge("load", "agent")
mb.add_edge("agent", "save")
mb.add_edge("save", END)

store = InMemoryStore()
saver = InMemorySaver()

# ⭐ BOTH. checkpointer = session continuity. store = relationship continuity.
mem_graph = mb.compile(checkpointer=saver, store=store)
print(mem_graph.get_graph().draw_mermaid())

---

## 9. The payoff — a NEW thread that still remembers you


In [ ]:
# --- Session 1 ---
cfg1 = {"configurable": {"thread_id": "session-1", "user_id": "ankit"}}
print("SESSION 1")
r = mem_graph.invoke(
    {"messages": [HumanMessage("I'm a backend engineer in Cologne. I'm vegetarian "
                               "and I'm training for a sub-60-minute 10K.")]},
    cfg1,
)
print("A:", r["messages"][-1].content[:200])

In [ ]:
# --- Session 2: brand-new thread_id. The CHECKPOINTER can't help here. ---
cfg2 = {"configurable": {"thread_id": "session-2", "user_id": "ankit"}}   # same user_id!
print("\nSESSION 2 (new thread — checkpointer has nothing)")
r = mem_graph.invoke({"messages": [HumanMessage("Suggest a dinner for tonight.")]}, cfg2)
print("A:", r["messages"][-1].content[:300])
print("\n^ It knows you're vegetarian. That came from the STORE, not the checkpointer.")

In [ ]:
# Inspect what's in the store
print("Stored memories for 'ankit':")
for item in store.search(("memories", "ankit")):
    print(f"  [{item.key[:8]}] {item.value['text']}")

# Namespace isolation check
print("\nMemories for a different user:", store.search(("memories", "someone-else")))

---

## 10. Semantic search in the Store

`InMemoryStore` does substring matching by default. Add an embedding index and `query=`
becomes a real vector search.


In [ ]:
# from langchain_openai import OpenAIEmbeddings
#
# semantic_store = InMemoryStore(
#     index={
#         "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
#         "dims": 1536,
#         "fields": ["text"],       # which value keys get embedded
#     }
# )
# # Now this is a true semantic lookup:
# semantic_store.search(("memories", "ankit"), query="food restrictions", limit=3)

print("Production stores: PostgresStore (pgvector) / AsyncPostgresStore")
print("API is identical — only the import and the constructor change.")

---

# 🟦 PART C — The 5-line version + context management

## 11. Memory on `create_react_agent`


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool


@tool
def log_run(distance_km: float, minutes: int) -> str:
    """Record a completed run."""
    return f"Logged {distance_km}km in {minutes}min ({minutes/distance_km:.1f} min/km)"


smart_agent = create_react_agent(
    llm,
    tools=[log_run],
    prompt="You are a running coach.",
    checkpointer=InMemorySaver(),    # ⭐ short-term
    store=InMemoryStore(),           # ⭐ long-term (tools can access it)
)

c = {"configurable": {"thread_id": "run-1", "user_id": "ankit"}}
smart_agent.invoke({"messages": [("user", "I ran 5km in 28 minutes today.")]}, c)
out = smart_agent.invoke({"messages": [("user", "How far did I run today?")]}, c)
print(out["messages"][-1].content)

---

## 12. Context management — do this *before* you need it

### Strategy 1: Trim (cheap, lossy)


In [ ]:
from langchain_core.messages import trim_messages

def pre_model_hook(state):
    """Runs before EVERY LLM call. Keeps the prompt within budget."""
    trimmed = trim_messages(
        state["messages"],
        strategy="last",              # keep the most recent
        token_counter=llm,            # use the model's real tokenizer
        max_tokens=2000,
        start_on="human",             # never start on a ToolMessage (API error!)
        end_on=("human", "tool"),
        include_system=True,          # always keep the system prompt
    )
    # `llm_input_messages` overrides what the LLM sees WITHOUT mutating stored state
    return {"llm_input_messages": trimmed}


trimmed_agent = create_react_agent(
    llm, tools=[log_run],
    pre_model_hook=pre_model_hook,
    checkpointer=InMemorySaver(),
)
print("Trimming agent ready — full history is still in the checkpoint, "
      "only the LLM's view is trimmed.")

### Strategy 2: Summarise (keeps the gist)


In [ ]:
from langchain_core.messages import RemoveMessage
from typing import TypedDict

class SummaryState(MessagesState):
    summary: str


def summarize_node(state: SummaryState) -> dict:
    """Condense old turns into one summary message, then DELETE them."""
    msgs = state["messages"]
    if len(msgs) <= 6:
        return {}

    prev = state.get("summary", "")
    prompt = (f"Previous summary:\n{prev}\n\nExtend it with these new messages:"
              if prev else "Summarize this conversation:")
    text = "\n".join(f"{m.type}: {m.content}" for m in msgs[:-4])
    summary = llm.invoke(f"{prompt}\n\n{text}").content

    # ⭐ RemoveMessage(id=...) tells add_messages to DELETE that message.
    # This is how you shrink state — you can't just return a shorter list.
    deletions = [RemoveMessage(id=m.id) for m in msgs[:-4]]
    return {"summary": summary, "messages": deletions}

print("RemoveMessage(id=...) is the ONLY way to delete from an add_messages list.")

---

## 13. Cheat Sheet

```
DEFINITION   state survives beyond one run -> identity over time
NOT A BASE   memory AUGMENTS ReAct/Planning/etc; it's rarely standalone

CHECKPOINTER  automatic | thread_id | whole state   | session continuity
STORE         explicit  | namespace | JSON docs     | relationship continuity
              ^ you need BOTH

INJECTION    def node(state, config, *, store: BaseStore)   <- names matter
NAMESPACE    ("memories", user_id)   = directory
KEY          "diet" (overwrite) vs uuid4() (accumulate)
DELETE       RemoveMessage(id=...)
GOTCHA #1    forgot thread_id
GOTCHA #2    new thread_id each turn
GOTCHA #3    no reducer -> state overwritten
```

**API essentials**

| Task | Code |
|---|---|
| Short-term | `builder.compile(checkpointer=InMemorySaver())` |
| Thread key | `{"configurable": {"thread_id": "x"}}` |
| Long-term | `builder.compile(store=InMemoryStore())` |
| Write fact | `store.put(("memories", uid), key, {"text": ...})` |
| Read facts | `store.search(("memories", uid), query="...")` |
| Inspect | `graph.get_state(config)` |
| Time travel | `graph.get_state_history(config)` → invoke with old config |
| Trim | `pre_model_hook` → `{"llm_input_messages": trimmed}` |
| Delete msgs | `RemoveMessage(id=m.id)` |
| Production | `PostgresSaver` + `PostgresStore` |

---

## 14. Decision Tree

```
Does the agent need to remember anything between invoke() calls?
├── NO ────────────────────────────► no memory needed (Pattern 1/2 as-is)
└── YES
    ├── Only within ONE conversation?
    │   └── YES ───────────────────► ✅ CHECKPOINTER only
    ├── Across conversations, same user?
    │   └── YES ───────────────────► ✅ CHECKPOINTER + STORE
    ├── Need human approval / rewind?
    │   └── YES ───────────────────► ✅ CHECKPOINTER (enables interrupts + time travel)
    └── Conversations getting long?
        ├── < 20 turns ────────────► trim_messages
        ├── 20–100 turns ──────────► summarization node
        └── > 100 turns ───────────► extract facts to Store, drop raw turns
```

---

## 15. Next

➡️ **Pattern 6 — Multi-Agent Systems**: several decision-makers instead of one.
